In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


X, y = make_classification(n_samples=200, n_features=5, n_informative=3, 
                           n_redundant=0, n_classes=2, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)  # BCELoss 需要 shape (N,1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, 1)  # 输出 1 个概率

    def forward(self, x):
        x = self.linear(x)
        x = F.relu(x)
        x = self.linear2(x)
        x = torch.sigmoid(x)  # Sigmoid 输出概率
        return x


model = BinaryClassifier(input_dim=X.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.BCELoss()


epochs = 100
for epoch in range(epochs):
    # 前向
    output = model(X_train)
    loss = criterion(output, y_train)

    # 反向
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 精度计算
    y_pred = (output > 0.5).float()#.float()：把 True/False 转成 0.0/1.0 的浮点张量，方便后续计算
    acc = torch.mean((y_pred == y_train).float())

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:03d}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")


model.eval()
with torch.no_grad():
    y_test_pred = (model(X_test) > 0.5).float()
    test_acc = torch.mean((y_test_pred == y_test).float())
    print("Test Accuracy:", test_acc.item())


Epoch 010, Loss: 0.5553, Accuracy: 0.7929
Epoch 020, Loss: 0.4840, Accuracy: 0.8643
Epoch 030, Loss: 0.4271, Accuracy: 0.8857
Epoch 040, Loss: 0.3822, Accuracy: 0.9000
Epoch 050, Loss: 0.3475, Accuracy: 0.9143
Epoch 060, Loss: 0.3213, Accuracy: 0.9143
Epoch 070, Loss: 0.3020, Accuracy: 0.9143
Epoch 080, Loss: 0.2875, Accuracy: 0.9214
Epoch 090, Loss: 0.2766, Accuracy: 0.9214
Epoch 100, Loss: 0.2681, Accuracy: 0.9214
Test Accuracy: 0.9166666865348816
